In [ ]:
# =============================================================================
# K-Threshold Analysis: Model 2 (Baseline2) & Model 3 (Proposed)
# This notebook evaluates the impact of different K values on retrieval performance.
# =============================================================================

from pathlib import Path
import sys
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

WORKING_DIR = Path.cwd()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / "code" / "01_supplier_identification").is_dir() else WORKING_DIR.parents[1]
IDENTIFICATION_CODE_DIR = PROJECT_ROOT / "code" / "01_supplier_identification"
sys.path.insert(0, str(IDENTIFICATION_CODE_DIR))

from histogram_baseline_retriever import Baseline2HistogramRetriever
from proposed_supplier_retriever import MultiLabelSupplierRetriever

# -----------------------------
# Common paths
# -----------------------------
BASE = PROJECT_ROOT
RESULT_DIR = BASE / "data" / "01_supplier_identification" / "main_split_70_30"  # Change to your dataset folder
EPOCH = 10

# Model 3 (embedding CSVs)
TRAIN_CSV = RESULT_DIR / "train_embeddings_epoch_010_with_metadata.csv"
TEST_CSV  = RESULT_DIR / "test_embeddings_epoch_010_with_metadata.csv"

# Model 2 (histogram CSVs)
HIST_DIR = RESULT_DIR / "histogram_baseline"
TRAIN_CSV_B2 = HIST_DIR / "train_histogram_features.csv"
TEST_CSV_B2  = HIST_DIR / "test_histogram_features.csv"

# -----------------------------
# K values to test
# -----------------------------
K_VALUES = [3, 5, 7, 9, 11, 13, 15, 20, 25, 30]

# -----------------------------
# Other settings
# -----------------------------
B2_SHAPE_SCALE = 10.0
INCLUDE_EXTRA_METRICS = True

# -----------------------------
# Utility function
# -----------------------------
def extract_metrics(model_name: str, k_value: int, metrics: dict, include_extra: bool = True) -> dict:
    """Extract metrics into a row dictionary."""
    row = {
        "Model": model_name,
        "K": k_value,
        "AvgPrecision": float(metrics.get("AvgPrecision", float("nan"))),
        "AvgRecall": float(metrics.get("AvgRecall", float("nan"))),
        "AvgF1": float(metrics.get("AvgF1", float("nan"))),
        "AvgAccuracy": float(metrics.get("AvgAccuracy", float("nan"))),
    }
    if include_extra:
        for k in ["MacroPurity@τ", "MicroPurity@τ", "ViolationRate_any@τ", "WrongProduct@1@τ"]:
            if k in metrics:
                row[k] = float(metrics[k])
    return row

# =============================================================================
# Model 2: Baseline2 (Histogram) - K Sweep
# =============================================================================

# Container for results
baseline2_results = []

if TRAIN_CSV_B2.exists() and TEST_CSV_B2.exists():
    print("[INFO] Running Baseline2 K-sweep...")
    print(f"  Train: {TRAIN_CSV_B2}")
    print(f"  Test:  {TEST_CSV_B2}")
    print(f"  K values: {K_VALUES}\n")
    
    for k_val in tqdm(K_VALUES, desc="Baseline2 K-sweep"):
        try:
            retriever = Baseline2HistogramRetriever(
                train_csv=str(TRAIN_CSV_B2),
                test_csv=str(TEST_CSV_B2),
                k_for_thr=k_val,
                metric_cols=None,
                shape_scale=B2_SHAPE_SCALE,
            )
            pred_sets, metrics = retriever.run_retrieval(
                verbose=False,
                require_same_component=False,
                require_same_assembly=False,
                log_top=0,
                margin_delta=0.01,
                debug_every=10**9,
                max_debug=0,
            )
            baseline2_results.append(
                extract_metrics("Baseline2", k_val, metrics, include_extra=INCLUDE_EXTRA_METRICS)
            )
        except Exception as e:
            print(f"[WARN] Baseline2 failed at K={k_val}: {e}")
            baseline2_results.append({
                "Model": "Baseline2",
                "K": k_val,
                "AvgPrecision": float("nan"),
                "AvgRecall": float("nan"),
                "AvgF1": float("nan"),
                "AvgAccuracy": float("nan"),
            })
    
    print(f"\n[INFO] Baseline2 K-sweep completed: {len(baseline2_results)} runs\n")
else:
    print("[WARN] Baseline2 skipped: histogram CSVs do not exist.\n")

# =============================================================================
# Model 3: Proposed (MultiLabel) - K Sweep
# =============================================================================

# Container for results
proposed_results = []

if TRAIN_CSV.exists() and TEST_CSV.exists():
    print("[INFO] Running Proposed model K-sweep...")
    print(f"  Train: {TRAIN_CSV}")
    print(f"  Test:  {TEST_CSV}")
    print(f"  K values: {K_VALUES}\n")
    
    for k_val in tqdm(K_VALUES, desc="Proposed K-sweep"):
        try:
            retriever = MultiLabelSupplierRetriever(
                train_csv=TRAIN_CSV,
                test_csv=TEST_CSV,
                k_for_threshold=k_val,
                require_same_component=False,
                require_same_assembly=False,
            )
            pred_sets, metrics = retriever.run_retrieval(verbose=False)
            proposed_results.append(
                extract_metrics("Proposed", k_val, metrics, include_extra=INCLUDE_EXTRA_METRICS)
            )
        except Exception as e:
            print(f"[WARN] Proposed model failed at K={k_val}: {e}")
            proposed_results.append({
                "Model": "Proposed",
                "K": k_val,
                "AvgPrecision": float("nan"),
                "AvgRecall": float("nan"),
                "AvgF1": float("nan"),
                "AvgAccuracy": float("nan"),
            })
    
    print(f"\n[INFO] Proposed model K-sweep completed: {len(proposed_results)} runs\n")
else:
    print("[WARN] Proposed model skipped: embedding CSVs do not exist.\n")

# =============================================================================
# Combined Results Table
# =============================================================================

# Combine all results
all_results = baseline2_results + proposed_results

if len(all_results) == 0:
    print("[WARN] No results to display.")
else:
    results_df = pd.DataFrame(all_results)
    
    # Reorder columns
    core_cols = ["Model", "K", "AvgPrecision", "AvgRecall", "AvgF1", "AvgAccuracy"]
    extra_cols = [c for c in results_df.columns if c not in core_cols]
    results_df = results_df[core_cols + extra_cols]
    
    # Display with formatting
    display(results_df.style.format(precision=4))

# =============================================================================
# Separate Tables by Model
# =============================================================================

if len(all_results) > 0:
    results_df = pd.DataFrame(all_results)
    
    # Baseline2 table
    if len(baseline2_results) > 0:
        print("=" * 80)
        print("Model 2: Baseline2 (Histogram) - K Threshold Analysis")
        print("=" * 80)
        baseline2_df = results_df[results_df["Model"] == "Baseline2"].reset_index(drop=True)
        display(baseline2_df.style.format(precision=4))
        print("\n")
    
    # Proposed table
    if len(proposed_results) > 0:
        print("=" * 80)
        print("Model 3: Proposed (MultiLabel) - K Threshold Analysis")
        print("=" * 80)
        proposed_df = results_df[results_df["Model"] == "Proposed"].reset_index(drop=True)
        display(proposed_df.style.format(precision=4))

# =============================================================================
# Performance Trends Visualization (Optional)
# =============================================================================

if len(all_results) > 0:
    results_df = pd.DataFrame(all_results)
    
    # Create subplots for each metric
    metrics_to_plot = ["AvgPrecision", "AvgRecall", "AvgF1", "AvgAccuracy"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    # Define colors (gray for Baseline2, yellow/gold for Proposed)
    color_baseline2 = '#3A3A3A'  # Dark gray
    color_proposed = '#FDB515'   # Golden yellow #DDA94B
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx]
        
        # Plot Baseline2
        if len(baseline2_results) > 0:
            b2_df = results_df[results_df["Model"] == "Baseline2"].sort_values("K")
            ax.plot(b2_df["K"], b2_df[metric], marker='o', color=color_baseline2, 
                   linewidth=2.5, markersize=7)
        
        # Plot Proposed
        if len(proposed_results) > 0:
            prop_df = results_df[results_df["Model"] == "Proposed"].sort_values("K")
            ax.plot(prop_df["K"], prop_df[metric], marker='s', color=color_proposed, 
                   linewidth=2.5, markersize=7)
        
        ax.set_xlabel("K (Threshold Parameter)", fontsize=12, fontweight='bold')
        ax.set_ylabel(metric, fontsize=12, fontweight='bold')
        ax.set_title(f"{metric} vs K", fontsize=13, fontweight='bold')
        ax.set_ylim(0.6, 1.01)  # Unified y-axis scale for fair comparison
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=11)
    
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] No results to visualize.")

# =============================================================================
# Best K Values Summary
# =============================================================================

if len(all_results) > 0:
    results_df = pd.DataFrame(all_results)
    
    best_k_summary = []
    
    for model_name in results_df["Model"].unique():
        model_df = results_df[results_df["Model"] == model_name]
        
        # Find best K for each metric
        for metric in ["AvgPrecision", "AvgRecall", "AvgF1", "AvgAccuracy"]:
            if metric in model_df.columns:
                best_idx = model_df[metric].idxmax()
                best_row = model_df.loc[best_idx]
                best_k_summary.append({
                    "Model": model_name,
                    "Metric": metric,
                    "Best_K": int(best_row["K"]),
                    "Value": float(best_row[metric])
                })
    
    if len(best_k_summary) > 0:
        print("=" * 80)
        print("Best K Values for Each Metric")
        print("=" * 80)
        best_k_df = pd.DataFrame(best_k_summary)
        display(best_k_df.style.format({"Value": "{:.4f}"}))
else:
    print("[INFO] No results available for best K analysis.")

# =============================================================================
# Export Results to CSV (Optional)
# =============================================================================

# Uncomment to save results
# if len(all_results) > 0:
#     output_path = RESULT_DIR / "K_Threshold_Analysis_Results.csv"
#     results_df = pd.DataFrame(all_results)
#     results_df.to_csv(output_path, index=False)
#     print(f"[INFO] Results saved to: {output_path}")